
# Workflow Practice

In this notebook, you’ll practice connecting to a SQLite database, creating tables from CSV files using Pandas, and writing SQL queries to explore the data.

The dataset comes from the [Bike Store Sample Database](https://www.kaggle.com/datasets/dillonmyrick/bike-store-sample-database) by Dillon Myrick. It models a fictional bike retailer with multiple stores, products, customers, and staff. Each table connects to others using foreign keys such as `customer_id`, `store_id`, and `product_id`.

You’ll:
- Connect to a local SQLite database
- Create tables using `pandas.to_sql()`
- Write and test SQL queries using `pd.read_sql()`

All of your work will take place directly in this notebook. Each question prompt is written below as a Markdown cell, followed by an empty code cell for you to write your query.



## Step 1: Connect to the Database

Run the following cell to connect to (or create) a SQLite database called `bike_store.db`.  
If the file doesn’t exist yet, SQLite will automatically create it.


In [6]:
import sqlite3
import pandas as pd
import os

In [7]:
connection = sqlite3.connect("bike_store.db")
connection


## Step 2: Create Tables from CSV Files

The `data/` folder contains one CSV file per table.  
Use `pandas.read_csv()` and `DataFrame.to_sql()` to load each file into your database.

You only need to do this once.  
After that, you’ll be able to run queries against your newly created tables.


In [4]:
# Example for one file
customers = pd.read_csv("data/customers.csv")
customers.to_sql("customers", connection, if_exists="replace", index=False)

1445

In [9]:
# Repeat for all other files in the data folder, or use a loop.
data_folder = "data"
for file in os.listdir(data_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(data_folder, file)

        table_name = os.path.splitext(file)[0]

        df = pd.read_csv(file_path)
        
        df.to_sql(table_name, connection, if_exists="replace", index=False)

        print(f"Loaded {file} into table '{table_name}'")


Loaded customers.csv into table 'customers'
Loaded categories.csv into table 'categories'
Loaded products.csv into table 'products'
Loaded orders.csv into table 'orders'
Loaded staffs.csv into table 'staffs'
Loaded order_items.csv into table 'order_items'
Loaded brands.csv into table 'brands'
Loaded stores.csv into table 'stores'
Loaded stocks.csv into table 'stocks'


### Verify Your Tables

Run a query to make sure your tables were created successfully.

In [10]:

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)


,name
0,customers
1,categories
2,products
3,orders
4,staffs
5,order_items
6,brands
7,stores
8,stocks


## Step 3: Test a Simple Query

Before starting the exercises, confirm your connection and tables are working by previewing the first few rows of the `customers` table.

In [11]:

pd.read_sql("SELECT * FROM customers LIMIT 5;", connection)


,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,Debra,Burks,None,debra.burks@yahoo.com,9273 Thorne Ave.,Orchard Park,NY,14127
1,2,Kasha,Todd,None,kasha.todd@yahoo.com,910 Vine Street,Campbell,CA,95008
2,3,Tameka,Fisher,None,tameka.fisher@aol.com,769C Honey Creek St.,Redondo Beach,CA,90278
3,4,Daryl,Spence,None,daryl.spence@aol.com,988 Pearl Lane,Uniondale,NY,11553
4,5,Charolette,Rice,(916) 381-6003,charolette.rice@msn.com,107 River Dr.,Sacramento,CA,95820


### Q1. List all customers and their cities.

Return the first name, last name, and city of each customer. Sort alphabetically by last name and then by first name.

In [14]:
# Your query here
pd.read_sql("""
    SELECT first_name, last_name, city
    FROM customers
    ORDER BY last_name, first_name;
    """, connection)

,first_name,last_name,city
0,Ester,Acevedo,San Lorenzo
1,Jamika,Acevedo,Ozone Park
2,Penny,Acevedo,Ballston Spa
3,Bettyann,Acosta,Lancaster
4,Shery,Acosta,Saratoga Springs
...,...,...,...
1440,Edda,Young,North Tonawanda
1441,Jasmin,Young,Helotes
1442,Alexandria,Zamora,Schenectady
1443,Jayme,Zamora,Springfield Gardens


### Q2. Show all products and their prices.

Display each product name along with its list price. Sort by price in descending order.

In [16]:
# Your query here
pd.read_sql(""" 
    SELECT product_name, list_price
    FROM products
    ORDER BY list_price DESC
    """, connection)

,product_name,list_price
0,Trek Domane SLR 9 Disc - 2018,11999.99
1,Trek Domane SLR 8 Disc - 2018,7499.99
2,Trek Silque SLR 8 Women's - 2017,6499.99
3,Trek Domane SL Frameset - 2018,6499.99
4,Trek Domane SL Frameset Women's - 2018,6499.99
...,...,...
316,Trek Kickster - 2018,159.99
317,Trek Boy's Kickster - 2015/2017,149.99
318,Trek Girl's Kickster - 2017,149.99
319,Sun Bicycles Lil Kitt'n - 2017,109.99


### Q3. Find all customers from California.

Return first name, last name, city, and state for all customers whose state is 'CA'. Sort alphabetically by last name.

In [19]:
# Your query h
pd.read_sql(""" 
    SELECT first_name, last_name, city, state
    FROM customers
    WHERE state = "CA" 
    ORDER BY last_name DESC
""", connection)

,first_name,last_name,city,state
0,Ollie,Zimmerman,Anaheim,CA
1,Yvone,Yates,San Pablo,CA
2,Joel,Wynn,San Diego,CA
3,Lucy,Woods,Palos Verdes Peninsula,CA
4,Darren,Witt,Coachella,CA
...,...,...,...,...
279,Selene,Austin,Duarte,CA
280,Twana,Arnold,Anaheim,CA
281,Sindy,Anderson,Pomona,CA
282,Jamaal,Albert,Torrance,CA


### Q4. Count how many products are in each category.

Return the category name and the number of products in that category. Sort from the highest count to the lowest.

In [24]:
# Your query here
pd.read_sql(""" 
    SELECT 
        c.category_name, 
        COUNT(p.product_id) AS product_count
    FROM   
        categories c
    JOIN
        Products p ON c.category_id = p.category_id
    GROUP BY
        c.category_name
    ORDER BY
        product_count DESC
""", connection)

,category_name,product_count
0,Cruisers Bicycles,78
1,Road Bikes,60
2,Mountain Bikes,60
3,Children Bicycles,59
4,Comfort Bicycles,30
5,Electric Bikes,24
6,Cyclocross Bicycles,10


### Q5. Find all orders placed in 2018.

List the order ID, order date, and customer ID for orders made during the year 2018. Sort by order date.

In [30]:
# Your query here
pd.read_sql(""" 
    SELECT order_id, order_date, customer_id
    FROM orders
    WHERE order_date BEtwEEN '2018-01-01' AND '2018-12-31'
    ORDER BY order_date;
""", connection)

,order_id,order_date,customer_id
0,1324,2018-01-01,862
1,1325,2018-01-01,68
2,1326,2018-01-01,567
3,1327,2018-01-02,1026
4,1328,2018-01-02,1083
...,...,...,...
287,1611,2018-09-06,6
288,1612,2018-10-21,3
289,1613,2018-11-18,1
290,1614,2018-11-28,135


### Q6. Show each order with its total number of items.

Join the `orders` and `order_items` tables. Group by order ID and return the number of items per order.

In [32]:
# Your query here
pd.read_sql(""" 
    SELECT 
        o.order_id,
        COUNT(oi.item_id) AS total_items
    FROM orders AS o
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    GROUP BY o.order_id
    ORDER BY o.order_id;
""", connection)

,order_id,total_items
0,1,5
1,2,2
2,3,2
3,4,1
4,5,3
...,...,...
1610,1611,3
1611,1612,5
1612,1613,2
1613,1614,3


### Q7. List total revenue per store.

Revenue = quantity * list_price * (1 - discount). Join `orders`, `order_items`, and `stores`, group by store name, and return total revenue.

In [41]:
# Your query here
pd.read_sql(""" 
    SELECT 
        st.store_name,
        '$' || printf('%.2f',SUM(oi.quantity * oi.list_price * (1 - oi.discount))) AS total_revenue
    FROM order_items AS oi
    JOIN orders AS o
        ON oi.order_id = o.order_id
    JOIN stores AS st
        ON o.store_id = st.store_id
    GROUP BY st.store_name
    ORDER BY total_revenue DESC;
""", connection)

,store_name,total_revenue
0,Rowlett Bikes,$867542.24
1,Baldwin Bikes,$5215751.28
2,Santa Cruz Bikes,$1605823.04


### Q8. Find the top 5 customers who spent the most overall.

Join `customers`, `orders`, and `order_items`. Sum the total spending per customer and return the top five spenders.

In [46]:
# Your query here
pd.read_sql(""" 
    SELECT 
        c.customer_id, 
        c.first_name,
        c.last_name,
        '$' || printf('%.2f',SUM(oi.quantity * oi.list_price * (1 - oi.discount))) AS total_spent
    FROM customers AS c
    JOIN orders AS o
        ON c.customer_id = o.customer_id
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.first_name, c.last_name
    ORDER BY SUM(oi.quantity * oi.list_price * (1 - oi.discount)) DESC
    LIMIT 5;
""", connection)

,customer_id,first_name,last_name,total_spent
0,94,Sharyn,Hopkins,$34807.94
1,10,Pamelia,Newman,$33634.26
2,75,Abby,Gamble,$32803.01
3,6,Lyndsey,Bean,$32675.07
4,16,Emmitt,Sanchez,$31925.89


### Q9. Show the best-selling product in each category.

Join `products`, `order_items`, and `categories`. For each category, identify the product with the highest total quantity sold.

In [48]:
# Your query here
pd.read_sql(""" 
    SELECT 
        p.product_name,
        c.category_name,
        SUM(oi.quantity) AS total_quantity_sold
    FROM products AS p
    JOIN order_items AS oi
        ON p.product_id = oi.product_id
    JOIN categories AS c
        ON p.category_id = c.category_id
    GROUP BY c.category_name, p.product_name
    HAVING total_quantity_sold = (
        SELECT MAX(sub.total_quantity)
        FROM (
            SELECT SUM(oi2.quantity) AS total_quantity
        FROM products AS p2
        JOIN order_items AS oi2
            ON p2.product_id = oi2.product_id
        WHERE p2.category_id = c.category_id
        GROUP BY p2.product_name
    ) AS sub
)
ORDER BY c.category_name;
""", connection)

,product_name,category_name,total_quantity_sold
0,Electra Girl's Hawaii 1 (20-inch) - 2015/2016,Children Bicycles,154
1,Electra Townie Original 7D - 2015/2016,Comfort Bicycles,148
2,Electra Cruiser 1 (24-Inch) - 2016,Cruisers Bicycles,157
3,Surly Straggler 650b - 2016,Cyclocross Bicycles,151
4,Trek Conduit+ - 2016,Electric Bikes,145
5,Surly Ice Cream Truck Frameset - 2016,Mountain Bikes,167
6,Trek Domane SLR 6 Disc - 2017,Road Bikes,43


### Q10. Identify the employees (staff) who processed the most orders.

Join `staffs` and `orders`. Count the number of orders handled by each staff member and return the results sorted by highest total.

In [50]:
# Your query here
pd.read_sql(""" 
    SELECT 
        s.staff_id,
        s.first_name,
        s.last_name,
        COUNT(o.order_id) AS total_orders
    FROM staffs AS s
    JOIN orders AS o
        ON s.staff_id = o.staff_id
    GROUP BY s.staff_id, s.first_name, s.last_name
    ORDER BY total_orders DESC;
""", connection)

,staff_id,first_name,last_name,total_orders
0,6,Marcelene,Boyer,553
1,7,Venita,Daniel,540
2,3,Genna,Serrano,184
3,2,Mireya,Copeland,164
4,8,Kali,Vargas,88
5,9,Layla,Terrell,86
